# LiDAR 정면 거리 측정

강의 예제 "LiDAR 활용하기 - 전방 거리 가져오기"를 바탕으로 수정했습니다.

## 하는 일
- RPLIDAR(통신 속도 460800)로 0° ±5° 범위의 거리를 모아 0.5초마다 중앙값(mm)을 출력합니다.
- 15초 동안 실행한 뒤 스스로 종료합니다.

## 원본에서 바꾼 점
- 스캔 시작 단계에서 `RPLidarException: Descriptor length mismatch`가 간헐적으로 발생합니다. 원인은 확인하지 못했고, 실패하면 `stop()`으로 정리한 뒤 최대 5번 재시도하도록 우회했습니다.
- 정면 값이 없을 때도 데이터가 오는지 알 수 있도록 1초마다 수신 개수와 마지막 측정값을 출력합니다.
- 종료할 때 정리 함수 하나가 실패해도 나머지가 실행되도록 했습니다.

## 확인한 것
- 수신 속도는 초당 약 5,100개였습니다.
- 정면에 손을 가까이 대면 값이 약 65~100mm까지 줄어들었고, 손을 떼면 다시 커졌습니다.
- 자로 잰 실제 거리와 출력값이 거의 일치했습니다. (실측 ___cm → 출력 약 ___mm)
- 스캔 시작은 이전에 2번 실패한 뒤 3번째에 성공한 적이 있었고, 마지막 실행에서는 1번 만에 성공했습니다. 실패 빈도와 원인은 아직 모릅니다.

In [3]:
import time
import statistics
from rplidar import RPLidar, RPLidarException

PORT = '/dev/ttyUSB0'
BAUDRATE = 460800
RUN_SECONDS = 15


def measures_with_retry(lidar, retries=5):
    """iter_measures()를 그대로 쓰되, 시작 단계에서 실패하면 정리 후 재시도."""
    for attempt in range(1, retries + 1):
        gen = lidar.iter_measures()
        try:
            first = next(gen)            # 스캔 시작 단계의 에러는 여기서 발생
        except RPLidarException as e:
            print(f"스캔 시작 실패 ({attempt}/{retries}): {e}")
            try:
                lidar.stop()             # 스캔 중지 + 입력 버퍼 정리
            except Exception:
                pass
            time.sleep(1.0)
            continue
        print(f"스캔 시작 성공 (시도 {attempt}회)")
        yield first
        yield from gen
        return
    raise RuntimeError("LiDAR 스캔 시작에 계속 실패했습니다")


print("LiDAR 연결 중...")
lidar = RPLidar(PORT, baudrate=BAUDRATE, timeout=2)

total = valid = 0
front_window = []
t_start = None
last_print = last_beat = time.time()

try:
    for new_scan, quality, angle, distance in measures_with_retry(lidar):
        now = time.time()
        if t_start is None:
            t_start = last_print = last_beat = now
        total += 1
        if distance > 0:
            valid += 1
            if angle <= 5 or angle >= 355:      # 정면 ±5도
                front_window.append(distance)

        if now - last_print > 0.5:               # 0.5초마다 정면 거리
            if front_window:
                print(f"Front distance: {statistics.median(front_window):.1f} mm "
                      f"(samples={len(front_window)})")
                front_window.clear()
            last_print = now

        if now - last_beat > 1.0:                # 1초마다 수신 상황
            print(f"[{now - t_start:4.1f}s] 수신 {total}개 (거리>0: {valid}개), "
                  f"마지막 angle={angle:.1f}도, dist={distance:.0f}mm")
            last_beat = now

        if now - t_start > RUN_SECONDS:
            break

except KeyboardInterrupt:
    print("\nStopping lidar...")

finally:
    for fn in (lidar.stop, lidar.stop_motor, lidar.disconnect):
        try:
            fn()
        except Exception:
            pass
    print(f"종료. 총 수신 {total}개, 거리>0 {valid}개")

LiDAR 연결 중...
스캔 시작 성공 (시도 1회)
Front distance: 392.2 mm (samples=65)
[ 1.0s] 수신 5055개 (거리>0: 4588개), 마지막 angle=168.4도, dist=1386mm
Front distance: 394.0 mm (samples=67)
Front distance: 383.8 mm (samples=63)
[ 2.0s] 수신 10142개 (거리>0: 9261개), 마지막 angle=336.3도, dist=100mm
Front distance: 100.8 mm (samples=68)
Front distance: 98.0 mm (samples=71)
[ 3.0s] 수신 15238개 (거리>0: 14023개), 마지막 angle=295.2도, dist=128mm
Front distance: 94.2 mm (samples=71)
Front distance: 94.0 mm (samples=72)
[ 4.0s] 수신 20331개 (거리>0: 18789개), 마지막 angle=318.8도, dist=100mm
Front distance: 95.0 mm (samples=71)
Front distance: 103.1 mm (samples=70)
[ 5.0s] 수신 25425개 (거리>0: 23558개), 마지막 angle=321.5도, dist=203mm
Front distance: 198.0 mm (samples=67)
Front distance: 219.5 mm (samples=72)
[ 6.0s] 수신 30520개 (거리>0: 28388개), 마지막 angle=321.5도, dist=216mm
Front distance: 244.0 mm (samples=69)
Front distance: 144.0 mm (samples=69)
[ 7.0s] 수신 35614개 (거리>0: 33177개), 마지막 angle=338.7도, dist=150mm
Front distance: 145.5 mm (samples=71)
Fr